In [ ]:

from pathlib import Path

import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
)

from lightgbm import LGBMClassifier, LGBMRegressor



random_state = 42

data_path = Path("../../dataset/US_Accidents/US_Accidents_March23.csv")

if not data_path.exists():
    matches = list(Path(".").rglob("US_Accidents_March23.csv"))
    if matches:
        data_path = matches[0]
    else:
        raise FileNotFoundError(
            "US_Accidents_March23.csv was not found. "
            "Put the CSV in dataset/US_Accidents/ or update data_path."
        )

print(f"Using dataset: {data_path}")


severity_features = [
    "Start_Lat",
    "Start_Lng",
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)",
    "Weather_Condition",
    "Junction",
    "Crossing",
    "Traffic_Signal",
    "Stop",
    "Railway",
    "Roundabout",
    "Bump",
    "Traffic_Calming",
]

precipitation_features = [
    "Wind_Speed(mph)",
    "Visibility(mi)",
    "Humidity(%)",
    "Weather_Condition",
    "Temperature(F)",
    "Pressure(in)",
]

severity_target = "Severity"
precipitation_target = "Precipitation(in)"

required_columns = list(dict.fromkeys(
    severity_features
    + precipitation_features
    + [severity_target, precipitation_target]
))


csv_dtype = {
    "Start_Lat": "float64",
    "Start_Lng": "float64",
    "Temperature(F)": "float64",
    "Humidity(%)": "float64",
    "Pressure(in)": "float64",
    "Visibility(mi)": "float64",
    "Wind_Speed(mph)": "float64",
    "Precipitation(in)": "float64",
    "Severity": "Int8",
    "Junction": "boolean",
    "Crossing": "boolean",
    "Traffic_Signal": "boolean",
    "Stop": "boolean",
    "Railway": "boolean",
    "Roundabout": "boolean",
    "Bump": "boolean",
    "Traffic_Calming": "boolean",
}


df = pd.read_csv(
    data_path,
    usecols=required_columns,
    dtype=csv_dtype,
    low_memory=True,
)

print("Loaded required columns only.")
print("Shape:", df.shape)
print("Approx RAM:", f"{df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("Duplicate rows:", df.duplicated().sum())


numeric_columns = [
    "Start_Lat",
    "Start_Lng",
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df["Weather_Condition"] = (
    df["Weather_Condition"]
    .fillna("Unknown")
    .astype("category")
)


boolean_columns = [
    "Junction",
    "Crossing",
    "Traffic_Signal",
    "Stop",
    "Railway",
    "Roundabout",
    "Bump",
    "Traffic_Calming",
]

for col in boolean_columns:
    df[col] = (
        df[col]
        .fillna(False)
        .astype("int8")
    )


df[severity_target] = pd.to_numeric(
    df[severity_target],
    errors="coerce",
)

df = df.dropna(
    subset=[severity_target]
).copy()

df[severity_target] = df[
    severity_target
].astype(int)


print("\nMissing values in model columns:")
print(
    df[
        severity_features + [severity_target]
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
)


def prepare_precipitation_data(data):
    x = data[precipitation_features].copy()

    for col in precipitation_features:
        if col != "Weather_Condition":
            x[col] = pd.to_numeric(
                x[col],
                errors="coerce",
            )

    x["Weather_Condition"] = (
        x["Weather_Condition"]
        .fillna("Unknown")
        .astype("category")
    )

    return x


available_mask = df[precipitation_target].notna()
missing_mask = ~available_mask

available_data = df.loc[
    available_mask
].copy()

missing_data = df.loc[
    missing_mask
].copy()

print(
    "\nPrecipitation available:",
    len(available_data),
)

print(
    "Precipitation missing:",
    len(missing_data),
)


train_precip, test_precip = train_test_split(
    available_data,
    test_size=0.20,
    random_state=random_state,
)

x_train_precip = prepare_precipitation_data(
    train_precip
)

x_test_precip = prepare_precipitation_data(
    test_precip
)

y_train_precip = train_precip[
    precipitation_target
]

y_test_precip = test_precip[
    precipitation_target
]

y_train_rain = (
    y_train_precip > 0
).astype(int)

y_test_rain = (
    y_test_precip > 0
).astype(int)


precip_classifier = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=random_state,
    n_jobs=-1,
    verbosity=-1,
)

precip_classifier.fit(
    x_train_precip,
    y_train_rain,
    categorical_feature=[
        "Weather_Condition"
    ],
)


rain_train = train_precip[
    train_precip[
        precipitation_target
    ] > 0
].copy()

x_rain_train = prepare_precipitation_data(
    rain_train
)

y_rain_train = np.log1p(
    rain_train[
        precipitation_target
    ]
)


precip_regressor = LGBMRegressor(
    objective="regression",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=random_state,
    n_jobs=-1,
    verbosity=-1,
)

precip_regressor.fit(
    x_rain_train,
    y_rain_train,
    categorical_feature=[
        "Weather_Condition"
    ],
)


rain_probability = (
    precip_classifier.predict_proba(
        x_test_precip
    )[:, 1]
)

rain_mask = (
    rain_probability >= 0.50
)

precip_pred = np.zeros(
    len(x_test_precip),
    dtype=float,
)

positive_idx = np.flatnonzero(
    rain_mask
)

if len(positive_idx) > 0:
    rain_amount_log = (
        precip_regressor.predict(
            x_test_precip.iloc[
                positive_idx
            ]
        )
    )

    precip_pred[
        positive_idx
    ] = np.expm1(
        rain_amount_log
    )

precip_pred = np.maximum(
    precip_pred,
    0,
)


mae = mean_absolute_error(
    y_test_precip,
    precip_pred,
)

rmse = np.sqrt(
    mean_squared_error(
        y_test_precip,
        precip_pred,
    )
)

print("\nPrecipitation validation")
print(
    f"MAE:  {mae:.6f}"
)

print(
    f"RMSE: {rmse:.6f}"
)

print(
    "Real zero %:      "
    f"{(y_test_precip == 0).mean() * 100:.2f}"
)

print(
    "Predicted zero %: "
    f"{(precip_pred == 0).mean() * 100:.2f}"
)


del (
    x_train_precip,
    x_test_precip,
    y_train_precip,
    y_test_precip,
    y_train_rain,
    y_test_rain,
    train_precip,
    test_precip,
    rain_train,
    x_rain_train,
    y_rain_train,
    rain_probability,
    rain_mask,
    precip_pred,
)

gc.collect()


x_full_precip = prepare_precipitation_data(
    available_data
)

y_full_precip = available_data[
    precipitation_target
]

y_full_rain = (
    y_full_precip > 0
).astype(int)


final_precip_classifier = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=random_state,
    n_jobs=-1,
    verbosity=-1,
)

final_precip_classifier.fit(
    x_full_precip,
    y_full_rain,
    categorical_feature=[
        "Weather_Condition"
    ],
)


rain_full = available_data[
    available_data[
        precipitation_target
    ] > 0
].copy()

x_rain_full = prepare_precipitation_data(
    rain_full
)

y_rain_full = np.log1p(
    rain_full[
        precipitation_target
    ]
)


final_precip_regressor = LGBMRegressor(
    objective="regression",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=random_state,
    n_jobs=-1,
    verbosity=-1,
)

final_precip_regressor.fit(
    x_rain_full,
    y_rain_full,
    categorical_feature=[
        "Weather_Condition"
    ],
)


del (
    x_full_precip,
    y_full_precip,
    y_full_rain,
    rain_full,
    x_rain_full,
    y_rain_full,
)

gc.collect()



if len(missing_data) > 0:

    x_missing = prepare_precipitation_data(
        missing_data
    )

    rain_probability_missing = (
        final_precip_classifier
        .predict_proba(
            x_missing
        )[:, 1]
    )

    rain_mask_missing = (
        rain_probability_missing >= 0.50
    )

    missing_predictions = np.zeros(
        len(x_missing),
        dtype=float,
    )

    positive_idx = np.flatnonzero(
        rain_mask_missing
    )

    if len(positive_idx) > 0:

        rain_amount_log = (
            final_precip_regressor.predict(
                x_missing.iloc[
                    positive_idx
                ]
            )
        )

        missing_predictions[
            positive_idx
        ] = np.expm1(
            rain_amount_log
        )

    missing_predictions = np.maximum(
        missing_predictions,
        0,
    )

    df.loc[
        missing_data.index,
        precipitation_target,
    ] = missing_predictions

    del (
        x_missing,
        rain_probability_missing,
        rain_mask_missing,
        missing_predictions,
        positive_idx,
    )

    gc.collect()


print(
    "\nRemaining precipitation missing:",
    df[
        precipitation_target
    ].isna().sum(),
)


# At this point the temporary missing-data frame is no longer needed.
del missing_data
gc.collect()


def prepare_severity_data(data):

    x = data[
        severity_features
    ].copy()

    for col in x.columns:
        if col != "Weather_Condition":
            x[col] = pd.to_numeric(
                x[col],
                errors="coerce",
            )

    x["Weather_Condition"] = (
        x["Weather_Condition"]
        .fillna("Unknown")
        .astype("category")
    )

    return x


x = prepare_severity_data(df)

y = df[
    severity_target
].astype(int)


x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=random_state,
    stratify=y,
)

print("\nAccident model")
print(
    "X_train:",
    x_train.shape,
)

print(
    "X_test :",
    x_test.shape,
)


accident_model = LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=random_state,
    n_jobs=-1,
    verbosity=-1,
)

accident_model.fit(
    x_train,
    y_train,
    categorical_feature=[
        "Weather_Condition"
    ],
)


y_pred = accident_model.predict(
    x_test
)

accuracy = accuracy_score(
    y_test,
    y_pred,
)

print(
    f"\nAccuracy: {accuracy * 100:.2f}%"
)

print(
    "\nClassification report:"
)

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0,
    )
)


df_final = df

print(
    "\nFinal dataset shape:",
    df_final.shape,
)

print(
    "Final precipitation missing:",
    df_final[
        precipitation_target
    ].isna().sum(),
)

print(
    "\nFinal precipitation statistics:"
)

print(
    df_final[
        precipitation_target
    ].describe()
)

print(
    "\nMemory-efficient training pipeline complete."
)



Using dataset: ..\..\dataset\US_Accidents\US_Accidents_March23.csv
Loaded required columns only.
Shape: (7728394, 18)
Approx RAM: 1018.0 MB
Duplicate rows: 643825

Missing values in model columns:
Precipitation(in)    2203586
Wind_Speed(mph)       571233
Visibility(mi)        177098
Humidity(%)           174144
Temperature(F)        163853
Pressure(in)          140679
Start_Lng                  0
Start_Lat                  0
Weather_Condition          0
Junction                   0
Crossing                   0
Traffic_Signal             0
Stop                       0
Railway                    0
Roundabout                 0
Bump                       0
Traffic_Calming            0
Severity                   0
dtype: int64

Precipitation available: 5524808
Precipitation missing: 2203586

Precipitation validation
MAE:  0.007812
RMSE: 0.090703
Real zero %:      90.36
Predicted zero %: 90.80

Remaining precipitation missing: 0

Accident model
X_train: (6182715, 17)
X_test : (1545679, 17)



In [6]:
from pathlib import Path
import json
import joblib

project_root = Path.cwd()


if not (project_root / "backend").exists():
    for candidate in [project_root, *project_root.parents]:
        if (candidate / "backend").exists():
            project_root = candidate
            break

model_dir = project_root / "backend" / "engines" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

accident_model_path = model_dir / "accident_model.joblib"
precip_classifier_path = model_dir / "precipitation_classifier.joblib"
precip_regressor_path = model_dir / "precipitation_regressor.joblib"
metadata_path = model_dir / "accident_model_metadata.json"

weather_categories = [
    str(x)
    for x in df["Weather_Condition"].cat.categories.tolist()
]


accident_bundle = {
    "model": accident_model,
    "features": list(severity_features),
    "target": severity_target,
    "weather_feature": "Weather_Condition",
    "weather_categories": weather_categories,
    "model_type": "LightGBM multiclass",
    "objective": "multiclass",
    "random_state": random_state,
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "dataset": str(data_path),
    "validation_accuracy": float(accuracy),
}

joblib.dump(
    accident_bundle,
    accident_model_path,
    compress=3,
)


precip_classifier_bundle = {
    "model": final_precip_classifier,
    "features": list(precipitation_features),
    "target": precipitation_target,
    "weather_feature": "Weather_Condition",
    "weather_categories": weather_categories,
    "model_type": "LightGBM binary rain classifier",
    "objective": "binary",
    "threshold": 0.50,
    "random_state": random_state,
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
}

joblib.dump(
    precip_classifier_bundle,
    precip_classifier_path,
    compress=3,
)

precip_regressor_bundle = {
    "model": final_precip_regressor,
    "features": list(precipitation_features),
    "target": precipitation_target,
    "target_transform": "log1p",
    "output_transform": "expm1",
    "weather_feature": "Weather_Condition",
    "weather_categories": weather_categories,
    "model_type": "LightGBM precipitation regressor",
    "objective": "regression",
    "random_state": random_state,
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
}

joblib.dump(
    precip_regressor_bundle,
    precip_regressor_path,
    compress=3,
)

metadata = {
    "status": "ready",
    "model_type": "LightGBM multiclass",
    "target": severity_target,
    "features": list(severity_features),
    "weather_categories_count": len(weather_categories),
    "classes": [
        int(x)
        for x in accident_model.classes_
    ],
    "validation_accuracy": float(accuracy),
    "random_state": random_state,
    "hyperparameters": {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "objective": "multiclass",
    },
    "precipitation_pipeline": {
        "classifier": str(precip_classifier_path),
        "regressor": str(precip_regressor_path),
        "rain_probability_threshold": 0.50,
        "target_transform": "log1p -> expm1",
    },
    "dataset": str(data_path),
    "notebook_pipeline": "US_Accident_prediction",
}

metadata_path.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 70)
print("MODELS SAVED FOR LA SMART ROUTE")
print("=" * 70)

print("\nAccident model:")
print(accident_model_path)

print("\nPrecipitation classifier:")
print(precip_classifier_path)

print("\nPrecipitation regressor:")
print(precip_regressor_path)

print("\nMetadata:")
print(metadata_path)

print("\nValidation accuracy:")
print(f"{accuracy * 100:.2f}%")

print("\nClasses:")
print(accident_model.classes_)

print("\nDone.")



MODELS SAVED FOR LA SMART ROUTE

Accident model:
c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction\backend\engines\models\accident_model.joblib

Precipitation classifier:
c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction\backend\engines\models\precipitation_classifier.joblib

Precipitation regressor:
c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction\backend\engines\models\precipitation_regressor.joblib

Metadata:
c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction\backend\engines\models\accident_model_metadata.json

Validation accuracy:
81.52%

Classes:
[1 2 3 4]

Done.
